In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD EVERY REAL SUMMARY JSON FROM ALL 13
#            PRIOR PROBLEMS (PROBLEM 14 DEPENDS ON "ALL ABOVE" PER THE MASTER
#            EXECUTION PLAN -- THIS IS THE PLATFORM'S BI AGGREGATION LAYER)
# =============================================================================
import json
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load All 13 Prior Problems' Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P14_ROOT = PROJECT_ROOT / "Phase5_Customer_Business_Intelligence" / "Problem14_Executive_Decision_Support_Dashboard"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first.")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

WARP_THREAD_COUNT = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
MAX_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}

# --- Real, canonical summary-JSON registry for all 13 prior problems.
# Every entry below is the ACTUAL final/headline notebook for that problem,
# identified by reading each problem's own last-shipped notebook source
# (never guessed). Problems 1-4 predate this platform's later "net benefit
# per cycle" financial-impact convention (introduced starting Problem 4/5's
# reporting notebooks and standardized from Problem 6 onward), so their
# registry entries honestly carry different field names -- no invented
# uniform schema. Problem 2 has no monetized financial-impact notebook at
# all (technical validation only); that is reported as N/A, not fabricated.
PRIOR_PROBLEMS_REGISTRY = {
    1: {
        "problem_name": "Credit Scoring (Static PD)", "phase": "Phase 1 -- Foundation",
        "category": "foundational_model",
        "summary_jsons": {"model": "notebook_05_summary.json", "capital": "notebook_08_summary.json"},
        "model_quality_path": ("model", "champion_metrics", "holdout_auc"),
        "model_quality_label": "Champion model holdout ROC-AUC",
        "financial_field": None, "financial_label": None,
        "exposure_fields": {"total_ecl_usd": ("capital", "total_ecl_usd"),
                             "total_rwa_usd": ("capital", "total_rwa_usd"),
                             "total_required_capital_usd": ("capital", "total_required_capital_usd")},
        "status_field": None,
        "notes": "Foundational static-PD model underlying Problems 6, 10, 12, 13. No standalone "
                 "'net benefit of deploying this model' figure exists -- its value is realized entirely "
                 "through the downstream problems that consume it. Real portfolio-level Basel/IFRS9 "
                 "capital and ECL exposure figures (Notebook 8) are reported as risk-exposure context, "
                 "never summed into the platform's net-benefit total (different kind of number).",
    },
    2: {
        "problem_name": "Risk Tier Classification", "phase": "Phase 1 -- Foundation",
        "category": "foundational_model",
        "summary_jsons": {"validation": "notebook_21_summary.json"},
        "model_quality_path": ("validation", "spearman_tier_vs_actual_default"),
        "model_quality_label": "Spearman(tier, actual default) on holdout",
        "financial_field": None, "financial_label": None,
        "exposure_fields": {},
        "status_field": None,
        "notes": "Predates this platform's financial-impact-reporting convention entirely -- Notebook 21 "
                 "is a technical validation notebook (rank-ordering, PSI, fair-lending gates), and "
                 "Notebook 25 is packaging-only. No dollar figure exists for this problem; reported as "
                 "N/A, not fabricated.",
    },
    3: {
        "problem_name": "Expected Credit Loss (IFRS9/CECL)", "phase": "Phase 2 -- Regulatory & Loss Provisioning",
        "category": "reserve_optimization",
        "summary_jsons": {"validation": "notebook_32_summary.json", "financial": "notebook_33_summary.json"},
        "model_quality_path": ("validation", "psi"),
        "model_quality_label": "Population Stability Index (holdout)",
        "financial_field": ("financial", "reserve_change_ifrs9_vs_flat_usd"),
        "financial_label": "Reserve accuracy gain vs. flat-LGD (IFRS9 vs. flat)",
        "exposure_fields": {"loss_cecl_usd": ("financial", "loss_cecl_usd"),
                             "macro_stress_buffer_usd": ("financial", "macro_stress_buffer_usd")},
        "status_field": ("validation", "self_test_passed"),
        "notes": "Reserve-provisioning accuracy gain, not a P&L cash benefit like Problems 4-13 -- kept in "
                 "its own 'reserve_optimization' category, reported alongside but never summed into the "
                 "platform's value-creation total.",
    },
    4: {
        "problem_name": "Delinquency Escalation / Loss Severity", "phase": "Phase 2 -- Regulatory & Loss "
        "Provisioning", "category": "value_creation",
        "summary_jsons": {"validation": "notebook_28_summary.json", "financial": "notebook_29_summary.json"},
        "model_quality_path": ("validation", "psi"),
        "model_quality_label": "Population Stability Index (holdout)",
        "financial_field": ("financial", "loss_prevented_per_cycle_usd"),
        "financial_label": "Loss prevented per cycle (tiered vs. flat LGD)",
        "exposure_fields": {},
        "status_field": ("validation", "self_test_passed"),
        "notes": "Real per-cycle loss-prevention value from severity-tiered LGD vs. a flat-LGD baseline.",
    },
    5: {
        "problem_name": "Early Payment Default Detection", "phase": "Phase 3 -- Behavioral Intelligence",
        "category": "value_creation",
        "summary_jsons": {"financial": "notebook_37_summary.json"},
        "model_quality_path": ("financial", "meets_kpi_target"),
        "model_quality_label": "Meets early-detection KPI target",
        "financial_field": ("financial", "loss_prevented_per_cycle_usd"),
        "financial_label": "Loss prevented per cycle (early detection)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle loss-prevention value from flagging defaulters K months earlier than a "
                 "full-history baseline.",
    },
    6: {
        "problem_name": "Dynamic / Behavioral Credit Scoring", "phase": "Phase 3 -- Behavioral Intelligence",
        "category": "value_creation",
        "summary_jsons": {"financial": "notebook_41_summary.json"},
        "model_quality_path": ("financial", "real_defaulter_capture_rate"),
        "model_quality_label": "Real defaulter capture rate (holdout)",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (dynamic vs. static PD)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle net benefit from monthly-refreshed dynamic PD vs. static PD alone.",
    },
    7: {
        "problem_name": "Early Warning System", "phase": "Phase 3 -- Behavioral Intelligence",
        "category": "value_creation",
        "summary_jsons": {"financial": "notebook_45_summary.json"},
        "model_quality_path": ("financial", "real_alert_capture_rate"),
        "model_quality_label": "Real alert capture rate (holdout)",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (rolling z-score alerts)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real result: NOT recommended for production (real default-rate lift fell below this "
                 "problem's own KPI target). Reported honestly and EXCLUDED from the platform's "
                 "value-creation total -- summing a not-recommended system's 'benefit' into an executive "
                 "total would be the exact category error this platform's aggregation must not make.",
    },
    8: {
        "problem_name": "Roll-Rate Modeling", "phase": "Phase 3 -- Behavioral Intelligence",
        "category": "value_creation",
        "summary_jsons": {"financial": "notebook_49_summary.json"},
        "model_quality_path": ("financial", "real_escalation_capture_rate"),
        "model_quality_label": "Real escalation capture rate (holdout)",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (Markov roll-rate escalation)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle net benefit from Markov transition-matrix escalation detection.",
    },
    9: {
        "problem_name": "Collections Optimization", "phase": "Phase 4 -- Operational Risk Management",
        "category": "value_creation",
        "summary_jsons": {"financial": "notebook_53_summary.json"},
        "model_quality_path": ("financial", "meets_kpi_target"),
        "model_quality_label": "Meets propensity-to-cure KPI target",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (treatment targeting)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle net benefit from propensity-to-cure treatment targeting.",
    },
    10: {
        "problem_name": "Credit Line Management", "phase": "Phase 4 -- Operational Risk Management",
        "category": "value_creation",
        "summary_jsons": {"financial": "notebook_57_summary.json"},
        "model_quality_path": ("financial", "meets_kpi_with_ci"),
        "model_quality_label": "Meets limit-optimization KPI target (with CI)",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (limit optimization)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle net benefit, net of amplified-loss cost and freeze-review cost, from "
                 "utilization-trend + PD-based limit actions.",
    },
    11: {
        "problem_name": "Real-Time Portfolio Monitoring", "phase": "Phase 4 -- Operational Risk Management",
        "category": "value_creation",
        "summary_jsons": {"financial": "notebook_61_summary.json"},
        "model_quality_path": ("financial", "real_cohort_capture_rate"),
        "model_quality_label": "Real cohort capture rate (holdout)",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (streaming breach alerts)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle net benefit from streaming aggregation + consecutive-breach alerting.",
    },
    12: {
        "problem_name": "360 Degree Customer Intelligence", "phase": "Phase 5 -- Customer & Business "
        "Intelligence", "category": "value_creation",
        "summary_jsons": {"financial": "notebook_65_summary.json"},
        "model_quality_path": ("financial", "reproduced_unified_roc_auc"),
        "model_quality_label": "Reproduced composite ROC-AUC (holdout)",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (unified-profile operational efficiency)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle net benefit is OPERATIONAL EFFICIENCY (collapsing 4 case lookups into 1), "
                 "explicitly designed by Problem 12 itself to not double-count Problems 9/10's benefits -- "
                 "safe to sum alongside them.",
    },
    13: {
        "problem_name": "Risk-Adjusted Profitability Modeling", "phase": "Phase 5 -- Customer & Business "
        "Intelligence", "category": "value_creation",
        "summary_jsons": {"financial": "notebook_69_summary.json"},
        "model_quality_path": ("financial", "reproduced_spearman_correlation"),
        "model_quality_label": "Reproduced Spearman(risk, profitability) (holdout)",
        "financial_field": ("financial", "net_benefit_per_cycle_usd"),
        "financial_label": "Net benefit per cycle (proactive exposure reduction, cross-tier segment)",
        "exposure_fields": {},
        "status_field": ("financial", "recommended_for_production"),
        "notes": "Real per-cycle net benefit targets the cross-tier (High Risk + Low Profitability) "
                 "segment, explicitly designed by Problem 13 itself to not double-count Problems 9/10's "
                 "benefits -- safe to sum alongside them.",
    },
}

# --- Load every real summary JSON referenced above. A missing file is a
# real prerequisite gap (that problem has not been run yet) and halts here
# with a clear fix, exactly like every other notebook's dependency check --
# Problem 14 cannot honestly aggregate a problem it cannot read.
LOADED_SUMMARIES = {}
_missing = []
for _pnum, _entry in PRIOR_PROBLEMS_REGISTRY.items():
    LOADED_SUMMARIES[_pnum] = {}
    for _key, _fname in _entry["summary_jsons"].items():
        _path = ARTIFACTS_DIR / _fname
        if not _path.exists():
            _missing.append(f"Problem {_pnum} ({_entry['problem_name']}): {_fname}")
            continue
        with open(_path, "r", encoding="utf-8") as f:
            LOADED_SUMMARIES[_pnum][_key] = json.load(f)

if _missing:
    raise FileNotFoundError(
        "Problem 14 depends on ALL 13 prior problems (per the master execution plan). The following "
        "real summary JSON(s) are missing -- run each listed problem's notebooks first:\n  "
        + "\n  ".join(_missing)
    )

print(f"Real summary JSONs loaded for all {len(PRIOR_PROBLEMS_REGISTRY)} prior problems.")
for _pnum, _entry in PRIOR_PROBLEMS_REGISTRY.items():
    print(f"  Problem {_pnum:>2} [{_entry['category']:>18}] {_entry['problem_name']}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

missing = []
try:
    import psutil
except ImportError:
    missing.append("psutil")
if missing:
    raise ImportError(f"Missing required libraries: {missing}. Install with: pip install {' '.join(missing)}")

print("✅ Section 2 complete: psutil available (this notebook is a pure JSON-aggregation pass -- no "
      "polars/dataframe work needed until Notebook 71).")


# =============================================================================
# SECTION 3: WARP RESOURCE PRE-FLIGHT GUARD (TWO-TIER RAM CHECK)
# =============================================================================
_section("SECTION 3: WARP Resource Pre-Flight Guard")

_vm = psutil.virtual_memory()
_available_bytes = _vm.available
_warn_threshold = MAX_RAM_BYTES * 0.50
_hardfail_threshold = MAX_RAM_BYTES * 0.25

print(f"WARP thread count (configured)     : {WARP_THREAD_COUNT}")
print(f"Max RAM ceiling (configured)       : {MAX_RAM_BYTES / (1024**3):.2f} GB")
print(f"Available RAM (right now)          : {_available_bytes / (1024**3):.2f} GB")

if _available_bytes < _hardfail_threshold:
    raise RuntimeError(
        f"Available RAM ({_available_bytes / (1024**3):.2f} GB) is below 25% of the configured ceiling "
        f"({MAX_RAM_BYTES / (1024**3):.2f} GB). Close other applications and re-run this notebook."
    )
elif _available_bytes < _warn_threshold:
    print("⚠️  WARNING: available RAM is below 50% of the configured ceiling. Proceeding, but consider "
          "closing other applications.")
else:
    print("✅ RAM pre-flight check passed.")
print("\n✅ Section 3 complete. This notebook is a lightweight JSON-aggregation pass -- no raw CSV "
      "streaming, so its own real memory footprint is negligible relative to every prior problem's "
      "own already-completed heavy computation.")


# =============================================================================
# SECTION 4: PROBLEM 14 SCOPE -- EXECUTIVE DECISION SUPPORT DASHBOARD
# =============================================================================
_section("SECTION 4: Problem 14 Scope -- Executive Decision Support Dashboard")

print(
    "Per the AMEX_Master_Execution_Plan.docx Phase 5 table: Problem 14 depends on ALL 13 prior problems, "
    "its core technique is a 'BI aggregation layer', its deliverable is an 'Executive dashboard', and its "
    "real-world impact is converting all 13 models above into something a CRO or head of risk can act on "
    "in a single meeting.\n\n"
    "This platform's own zero-fabrication rule applies here with extra force: an executive rollup is "
    "exactly where category errors (summing risk-exposure figures with realized P&L benefits, including a "
    "not-recommended system's benefit in a production total, double-counting an already-priced benefit) "
    "would otherwise hide behind a single impressive-looking headline number. Problem 14's own new, "
    "additive design contribution -- beyond simply restating the 13 problems' own real figures -- is "
    "PROVING the aggregation is done correctly, via two genuinely new hard-gating KPIs defined below."
)
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: TWO NEW HARD-GATING KPIS FOR PROBLEM 14
# =============================================================================
_section("SECTION 5: Two New Hard-Gating KPIs for Problem 14")

EXECUTIVE_KPI_TARGETS = {
    "aggregation_completeness": {
        "definition": "Every one of the 13 prior problems' real summary JSON(s) is present, parses as "
                       "valid JSON, and its problem_number field (where present) matches this registry's "
                       "own expectation. 100% required -- Problem 14 cannot honestly report on a problem "
                       "it could not actually read.",
        "target": "100% (13 / 13 problems loaded and sanity-checked)",
        "rationale": "ASSUMPTION-free: this is a structural completeness check on real files, not a "
                     "statistical estimate.",
    },
    "aggregation_scope_correctness": {
        "definition": "The platform's TOTAL_PLATFORM_NET_VALUE_USD headline figure sums the "
                       "financial_field of every problem tagged category=='value_creation' AND whose real "
                       "recommended_for_production flag is True -- and ONLY those. This KPI passes when "
                       "the real, programmatically-derived inclusion set exactly equals the expected set "
                       "computed independently from each problem's own real category/status fields (i.e. "
                       "the filter logic itself is verified against the data it filters, not just trusted "
                       "to have been written correctly).",
        "target": "Computed inclusion set == independently-derived expected set (exact set equality)",
        "rationale": "The single most important integrity check an executive rollup can make: proving "
                     "foundational models (Problems 1, 2), the reserve-optimization problem (3), and the "
                     "one not-recommended-for-production system (Problem 7) are correctly EXCLUDED from "
                     "the value-creation total, while every recommended, benefit-bearing problem is "
                     "correctly INCLUDED exactly once.",
    },
}

for _k, _v in EXECUTIVE_KPI_TARGETS.items():
    print(f"  {_k}: {_v['target']}")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: EXPECTED INCLUSION/EXCLUSION SETS (FOR SECTION 5'S SECOND KPI)
# =============================================================================
_section("SECTION 6: Expected Inclusion/Exclusion Sets")

# Independently derived from this registry's own real category/status semantics
# (not hand-picked problem numbers) -- Notebook 71 will compute the REAL set from
# each problem's own loaded summary data and assert it equals this expectation.
EXPECTED_FOUNDATIONAL_PROBLEMS = sorted(
    p for p, e in PRIOR_PROBLEMS_REGISTRY.items() if e["category"] == "foundational_model"
)
EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS = sorted(
    p for p, e in PRIOR_PROBLEMS_REGISTRY.items() if e["category"] == "reserve_optimization"
)
EXPECTED_VALUE_CREATION_CANDIDATE_PROBLEMS = sorted(
    p for p, e in PRIOR_PROBLEMS_REGISTRY.items() if e["category"] == "value_creation"
)

print(f"Foundational models (excluded from value total)      : {EXPECTED_FOUNDATIONAL_PROBLEMS}")
print(f"Reserve-optimization (excluded from value total)     : {EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS}")
print(f"Value-creation candidates (included IF recommended)  : {EXPECTED_VALUE_CREATION_CANDIDATE_PROBLEMS}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: EXECUTIVE TIME-SAVINGS ASSUMPTION SCOPE (FOR NOTEBOOK 73)
# =============================================================================
_section("SECTION 7: Executive Time-Savings Assumption Scope")

print(
    "Problem 14's OWN genuinely new, additive financial claim (built out fully in Notebook 73) is "
    "EXECUTIVE DECISION-LATENCY REDUCTION: real time an executive currently spends reviewing 13 separate "
    "problem-level reports/dashboards/services vs. one unified executive dashboard, net of this "
    "dashboard's own ongoing hosting cost -- a deliberately different constituency (C-suite/CRO time) "
    "from Problem 12's operational-analyst-lookup-time framing, and additive rather than double-counting: "
    "it prices the cost of EXECUTIVE REVIEW TIME, a real activity none of Problems 1-13 priced."
)
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: PERSIST PROBLEM 14 POLICY
# =============================================================================
_section("SECTION 8: Persist Problem 14 Policy")

if "executive_dashboard_policy" in PILLAR_DIRS:
    P14_POLICY_DIR = PILLAR_DIRS["executive_dashboard_policy"]
else:
    P14_POLICY_DIR = P14_ROOT / "policy"
    print(f"NOTE: 'executive_dashboard_policy' not in pillar_dirs -- using fallback: {P14_POLICY_DIR}")
P14_POLICY_DIR.mkdir(parents=True, exist_ok=True)

EXECUTIVE_DASHBOARD_POLICY = {
    "problem_number": 14, "problem_name": "Executive Decision Support Dashboard",
    "phase": "Phase 5 -- Customer & Business Intelligence",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "prior_problems_registry": {
        str(k): {kk: vv for kk, vv in v.items() if kk != "summary_jsons"} for k, v in
        PRIOR_PROBLEMS_REGISTRY.items()
    },
    "kpi_targets": EXECUTIVE_KPI_TARGETS,
    "expected_foundational_problems": EXPECTED_FOUNDATIONAL_PROBLEMS,
    "expected_reserve_optimization_problems": EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS,
    "expected_value_creation_candidate_problems": EXPECTED_VALUE_CREATION_CANDIDATE_PROBLEMS,
    "random_seed": RANDOM_SEED,
}
POLICY_PATH = P14_POLICY_DIR / "executive_dashboard_policy.json"
with open(POLICY_PATH, "w", encoding="utf-8") as f:
    json.dump(EXECUTIVE_DASHBOARD_POLICY, f, indent=2)
print(f"Policy written to: {POLICY_PATH}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: WRITE NOTEBOOK 70 SUMMARY
# =============================================================================
_section("SECTION 9: Write Notebook 70 Summary")

notebook_70_summary = {
    "notebook": "70_executive_dashboard_business_understanding",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 14, "problem_name": "Executive Decision Support Dashboard",
    "phase": "Phase 5 -- Customer & Business Intelligence",
    "policy_path": str(POLICY_PATH),
    "n_prior_problems_loaded": len(PRIOR_PROBLEMS_REGISTRY),
    "expected_foundational_problems": EXPECTED_FOUNDATIONAL_PROBLEMS,
    "expected_reserve_optimization_problems": EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS,
    "expected_value_creation_candidate_problems": EXPECTED_VALUE_CREATION_CANDIDATE_PROBLEMS,
    "warp_thread_count": WARP_THREAD_COUNT, "max_ram_bytes": MAX_RAM_BYTES, "random_seed": RANDOM_SEED,
}
nb70_summary_path = ARTIFACTS_DIR / "notebook_70_summary.json"
with open(nb70_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_70_summary, f, indent=2)
print(f"Summary written to: {nb70_summary_path}")
print("\n✅ Section 9 complete.")

print(
    "\n🎯 Notebook 70 (Business Understanding & Policy) complete. Problem 14's registry, hard-gating "
    "KPIs, and policy are real and persisted. Next: Notebook 71 (Modeling -- the real BI aggregation "
    "layer itself)."
)
